# Roster Parser

This notebook is designed to help you understand the existing code for parsing school rosters from scanned PDF files.

## Objective
The goal is to extract structured data from scanned PDF rosters and save it as JSON files. The JSON structure should match the pipeline's expected schema.

You will find the existing code, its explanation, and an example of how to use it. Feel free to suggest improvements or alternative approaches.

## Code Overview
The `roster_parser.py` script is designed to handle the following tasks:

1. **Excel Parsing**: Extracts data from Excel files in various formats.
2. **PDF Parsing**: Uses GPT-4o vision to extract data from scanned PDF rosters.
3. **Output Schema**: Produces a JSON file with the following structure:

```json
{
  "school": "学校名",
  "year": 3,
  "classes": {
    "1": {
      "teacher": "山田 祐輔",
      "student_count": 12,
      "students": [
        {"number": 1, "name": "飯田 優香", "furigana": "いいだ ゆうか", "gender": ""},
        {"number": 2, "name": "飯塚 朱李", "furigana": "いいづか あかり", "gender": ""}
      ]
    }
  },
  "warnings": []
}
```

4. **Validation**: Ensures data integrity by checking for missing or duplicate attendance numbers.

## Code

In [ ]:
"""
roster_parser.py – Universal roster → roster.json converter
============================================================
Supports Excel (.xlsx) and scanned PDF rosters.

Excel formats:
  Format A ("千早高-style")
    • Sheet named "名票" (or "アルバム委員")
    • HR blocks repeated every 11 columns
    • Row 2 contains HR labels (HR31, HR32 …)
    • Data rows start at row 6

  Format B ("船橋啓明高-style")
    • Paired-class sheets ("3AB", "3CD", …)
    • Row 1: "３Ａ担任：Teacher Name"  |  "３Ｂ担任：…"
    • Row 2: Headers (No, 氏名, ふりがな, 性)
    • Data rows start at row 3

PDF format:
  Uses GPT-4o vision to extract structured data from scanned roster images.
  Requires an OpenAI API key (passed via --openai-key or OPENAI_API_KEY env var).

Output schema (roster.json):
{
  "school": "...",
  "year": 3,
  "classes": {
     "A": {
        "teacher": "片江 佳奈子",
        "student_count": 39,
        "students": [
           {"number": 1, "name": "青木 伸篤", "furigana": "あおき のぶあつ", "gender": "男"},
           ...
        ]
     },
     ...
  }
}
"""

from __future__ import annotations

import base64
import io
import json
import logging
import os
import re
from pathlib import Path
from typing import Any

import openpyxl

logger = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────────────────────────

_FULL_TO_HALF = str.maketrans(
    "０１２３４５６７８９ＡＢＣＤＥＦＧＨＩＪＫＬＭＮＯＰＱＲＳＴＵＶＷＸＹＺ"
    "ａｂｃｄｅｆｇｈｉｊｋｌｍｎｏｐｑｒｓｔｕｖｗｘｙｚ",
    "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "abcdefghijklmnopqrstuvwxyz",
)


def _normalise(text: str) -> str:
    """Full-width → half-width, strip whitespace."""
    if text is None:
        return ""
    return str(text).translate(_FULL_TO_HALF).strip()


def _extract_class_id(raw: str) -> str | None:
    """
    Extract the class identifier, preserving the original format.
    '３Ａ担任：…' → 'A'  (letter-based school)
    'HR31'      → '1'   (number-based school, class 1)
    '3AB'       → parse later.
    """
    raw = _normalise(raw)
    # "3A担任" pattern — already a letter
    m = re.search(r"(\d)\s*([A-Za-z])\s*担任", raw)
    if m:
        return m.group(2).upper()
    # "HR3X" pattern  (千早高 uses numeric suffixes: HR31→class1, HR32→class2 …)
    m = re.search(r"HR\d(\d)", raw)
    if m:
        return m.group(1)  # preserve original number as string
    return None


def _extract_teacher(raw: str) -> str | None:
    """'３Ａ担任：片江　佳奈子' → '片江 佳奈子'"""
    raw = _normalise(raw)
    m = re.search(r"担任[：:]\s*(.+)", raw)
    if m:
        name = m.group(1).strip()
        # collapse multiple spaces / full-width spaces
        name = re.sub(r"[\s　]+", " ", name)
        return name
    return None


# ──────────────────────────────────────────────────────────────────
# Format B — Paired-class sheets  (船橋啓明高-style)
# ──────────────────────────────────────────────────────────────────

_PAIRED_SHEET_RE = re.compile(r"^(\d)([A-Za-z]{2})$")


def _detect_format_b(wb: openpyxl.Workbook) -> bool:
    """Return True if any sheet name matches '3AB', '3CD' etc."""
    for name in wb.sheetnames:
        if _PAIRED_SHEET_RE.match(_normalise(name)):
            return True
    return False


def _parse_format_b(wb: openpyxl.Workbook) -> dict:
    classes: dict[str, dict] = {}

    for sheet_name in wb.sheetnames:
        norm = _normalise(sheet_name)
        m = _PAIRED_SHEET_RE.match(norm)
        if not m:
            continue
        ws = wb[sheet_name]
        _year = int(m.group(1))
        pair = m.group(2).upper()  # "AB", "CD" …

        # ── Left class (cols A-D) ──
        left_header = str(ws.cell(1, 1).value or "")
        left_letter = _extract_class_id(left_header)
        left_teacher = _extract_teacher(left_header)
        left_count_raw = ws.cell(1, 5).value  # col E
        left_count = int(left_count_raw) if left_count_raw else None

        # ── Right class (cols G-J) ──
        right_header = str(ws.cell(1, 7).value or "")
        right_letter = _extract_class_id(right_header)
        right_teacher = _extract_teacher(right_header)
        right_count_raw = ws.cell(1, 11).value  # col K
        right_count = int(right_count_raw) if right_count_raw else None

        # Fallback letters from sheet name
        if not left_letter:
            left_letter = pair[0]
        if not right_letter:
            right_letter = pair[1]

        # ── Read students ──
        for letter, num_col, name_col, furi_col, gender_col, teacher, count in [
            (left_letter, 1, 2, 3, 4, left_teacher, left_count),
            (right_letter, 7, 8, 9, 10, right_teacher, right_count),
        ]:
            students = []
            seen_numbers: set[int] = set()
            consecutive_empty = 0
            for row in range(3, ws.max_row + 1):
                num_val = ws.cell(row, num_col).value
                name_val = ws.cell(row, name_col).value
                if num_val is None or name_val is None:
                    consecutive_empty += 1
                    if consecutive_empty >= 3:
                        break  # stop after 3 consecutive empty rows (end of roster)
                    continue
                consecutive_empty = 0
                try:
                    num = int(num_val)
                except (ValueError, TypeError):
                    continue
                if num < 1:
                    continue
                if num in seen_numbers:
                    continue  # skip duplicate attendance numbers
                seen_numbers.add(num)
                students.append({
                    "number": num,
                    "name": re.sub(r"[\s　]+", " ", str(name_val).strip()),
                    "furigana": re.sub(r"[\s　]+", " ", str(ws.cell(row, furi_col).value or "").strip()),
                    "gender": str(ws.cell(row, gender_col).value or "").strip(),
                })
            students.sort(key=lambda s: s["number"])
            classes[letter] = {
                "teacher": teacher,
                "student_count": count or len(students),
                "students": students,
            }

    return {"year": _year if classes else 3, "classes": classes}


# ──────────────────────────────────────────────────────────────────
# Format A — Wide HR-block layout  (千早高-style)
# ──────────────────────────────────────────────────────────────────

def _detect_format_a(wb: openpyxl.Workbook) -> bool:
    return "名票" in wb.sheetnames or "アルバム委員" in wb.sheetnames


def _parse_format_a(wb: openpyxl.Workbook) -> dict:
    ws = wb["名票"] if "名票" in wb.sheetnames else wb["アルバム委員"]

    # Read the column stride from "定義" if available
    stride = 11  # default
    if "定義" in wb.sheetnames:
        dws = wb["定義"]
        for row in dws.iter_rows(min_row=1, max_row=20, values_only=False):
            for cell in row:
                if cell.value and "１クラスあたり列数" in str(cell.value):
                    # value is in the cell 2 cols to the right
                    val = dws.cell(cell.row, cell.column + 2).value
                    if val:
                        stride = int(val)

    # Scan row 2 for HR labels → determine block start columns
    blocks: list[tuple[str, int]] = []  # (class_letter, start_col_0based)
    for col in range(1, ws.max_column + 1):
        val = ws.cell(2, col).value
        if val and re.search(r"HR\d+", str(val)):
            letter = _extract_class_id(str(val))
            if letter:
                # The number column is one col before the HR label col
                # In the data: E=number, F=name  where F is the HR-label column
                blocks.append((letter, col))

    # Also try to find teacher from デザイン sheet
    teachers: dict[str, str] = {}
    if "デザイン" in wb.sheetnames:
        dws = wb["デザイン"]
        for row in dws.iter_rows(min_row=1, max_row=dws.max_row, values_only=False):
            for cell in row:
                if cell.value and "担任" in str(cell.value):
                    # Teacher name might be nearby — check next row same col, or adjacent
                    pass  #千早高 only has the label "担任", not the actual name here

    classes: dict[str, dict] = {}

    for letter, name_col in blocks:
        # name_col is where the HR label sits = also the Name column in data rows
        num_col = name_col - 1         # attendance number
        gender_col = name_col + 1      # gender
        furi_col = name_col + 2        # furigana

        students = []
        for row in range(3, ws.max_row + 1):
            num_val = ws.cell(row, num_col).value
            name_val = ws.cell(row, name_col).value
            if num_val is None or name_val is None:
                continue
            try:
                num = int(num_val)
            except (ValueError, TypeError):
                continue
            if num < 1:
                continue
            students.append({
                "number": num,
                "name": re.sub(r"[\s　]+", " ", str(name_val).strip()),
                "furigana": re.sub(r"[\s　]+", " ", str(ws.cell(row, furi_col).value or "").strip()),
                "gender": str(ws.cell(row, gender_col).value or "").strip(),
            })

        students.sort(key=lambda s: s["number"])
        classes[letter] = {
            "teacher": teachers.get(letter),
            "student_count": len(students),
            "students": students,
        }

    return {"year": 3, "classes": classes}


# ──────────────────────────────────────────────────────────────────
# Generic fallback — scan all sheets for (number, name) columns
# ──────────────────────────────────────────────────────────────────

def _parse_generic(wb: openpyxl.Workbook) -> dict:
    """
    Best-effort: look for sheets that contain columns with headers
    matching No/番号 and 氏名/名前.
    """
    classes: dict[str, dict] = {}
    class_idx = 0

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        if ws.max_row < 3:
            continue

        # Try to find header row (first 5 rows)
        num_col = name_col = furi_col = gender_col = None
        header_row = None
        for r in range(1, min(6, ws.max_row + 1)):
            for c in range(1, min(30, ws.max_column + 1)):
                val = _normalise(str(ws.cell(r, c).value or ""))
                if val in ("no", "no.", "番号"):
                    num_col = c
                    header_row = r
                elif "氏" in val and "名" in val:
                    name_col = c
                elif "ふりがな" in val or "フリガナ" in val.upper():
                    furi_col = c
                elif val in ("性", "性別"):
                    gender_col = c

        if num_col is None or name_col is None or header_row is None:
            continue

        # Check row 1 for teacher
        teacher = None
        for c in range(1, min(15, ws.max_column + 1)):
            t = _extract_teacher(str(ws.cell(1, c).value or ""))
            if t:
                teacher = t
                break

        letter_from_header = None
        for c in range(1, min(15, ws.max_column + 1)):
            l = _extract_class_id(str(ws.cell(1, c).value or ""))
            if l:
                letter_from_header = l
                break

        letter = letter_from_header or str(class_idx + 1)
        class_idx += 1

        students = []
        for row in range(header_row + 1, ws.max_row + 1):
            num_val = ws.cell(row, num_col).value
            name_val = ws.cell(row, name_col).value
            if num_val is None or name_val is None:
                continue
            try:
                num = int(num_val)
            except (ValueError, TypeError):
                continue
            if num < 1:
                continue
            students.append({
                "number": num,
                "name": re.sub(r"[\s　]+", " ", str(name_val).strip()),
                "furigana": re.sub(r"[\s　]+", " ", str(ws.cell(row, furi_col).value or "").strip()) if furi_col else "",
                "gender": str(ws.cell(row, gender_col).value or "").strip() if gender_col else "",
            })

        if students:
            students.sort(key=lambda s: s["number"])
            classes[letter] = {
                "teacher": teacher,
                "student_count": len(students),
                "students": students,
            }

    return {"year": 3, "classes": classes}


# ──────────────────────────────────────────────────────────────────
# PDF parsing via GPT-4o vision
# ──────────────────────────────────────────────────────────────────

_PDF_ROSTER_PROMPT = """
You are given a scanned image of a Japanese school student roster (名票 / 学年名票).

Extract ALL classes and ALL students visible in this image.

Each class section typically shows:
- A header like "第3学年1組" or "3年A組" (grade + class number/letter)
- A homeroom teacher line like "担任 山田 太郎"
- A numbered list of students with: attendance number, full name (kanji), furigana (hiragana reading)
- Optionally a gender column (男/女)

Rules:
- Extract EVERY student without exception. Never skip, merge, or omit any row.
- Attendance numbers are sequential integers (1, 2, 3, ...). If a number cell is empty, infer it from position.
- Keep original kanji names EXACTLY as printed, including correct spacing between surname and given name.
- Furigana MUST match the kanji name: provide both surname reading AND given name reading separated by a space.
  Example: if name is "山田 太郎", furigana must be "やまだ たろう" (not just "たろう").
- For class identification: preserve the ORIGINAL identifier from the document.
  If the class uses a NUMBER (e.g. "第3学年1組" or "3年1組"), use that number as
  the JSON key: "1", "2", "3", etc.
  If the class uses a LETTER (e.g. "3年A組"), use that letter: "A", "B", etc.
- Gender: ONLY populate this field if the document has an explicit gender column (男/女 printed next to each student).
  If there is no gender column, set gender to "" for every student. Do NOT infer gender from names.
- If furigana is not visible or unclear, use "" (empty string) — do NOT guess or fabricate.
- Be careful with rare or unusual kanji. Transcribe exactly what is printed.

Return STRICT JSON only. No markdown, no explanation, no ``` blocks.

{
  "classes": {
    "1": {
      "teacher": "teacher full name or null",
      "students": [
        {"number": 1, "name": "山田 太郎", "furigana": "やまだ たろう", "gender": "男"},
        {"number": 2, "name": "鈴木 花子", "furigana": "すずき はなこ", "gender": "女"},
        ...
      ]
    },
    "2": { ... },
    ...
  }
}
""".strip()


def _render_pdf_pages(pdf_path: Path, dpi: int = 200) -> list[bytes]:
    """Render each PDF page as a PNG byte buffer."""
    import fitz  # PyMuPDF

    doc = fitz.open(str(pdf_path))
    pages: list[bytes] = []
    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)
    for page in doc:
        pix = page.get_pixmap(matrix=mat)
        pages.append(pix.tobytes("png"))
    doc.close()
    return pages


def _image_to_data_url(png_bytes: bytes) -> str:
    """Convert PNG bytes to a data URL for the OpenAI API."""
    b64 = base64.b64encode(png_bytes).decode("utf-8")
    return f"data:image/png;base64,{b64}"


def _repair_truncated_json(raw: str) -> dict | None:
    """
    Attempt to repair a truncated JSON string from GPT by closing
    open brackets/braces. Works for the common case where the response
    was cut mid-array or mid-object.
    """
    # Strip trailing incomplete key-value pairs or strings
    # Remove trailing comma or partial entry
    cleaned = raw.rstrip()
    # Remove trailing incomplete string (unclosed quote)
    if cleaned.count('"') % 2 != 0:
        # Find last complete quote pair
        last_quote = cleaned.rfind('"')
        if last_quote > 0:
            cleaned = cleaned[:last_quote]
            # Now remove the incomplete entry back to the last comma or bracket
            last_good = max(cleaned.rfind(','), cleaned.rfind('['), cleaned.rfind('{'))
            if last_good > 0:
                cleaned = cleaned[:last_good + 1]

    # Remove trailing comma
    cleaned = cleaned.rstrip().rstrip(',')

    # Count open brackets/braces and close them
    open_braces = cleaned.count('{') - cleaned.count('}')
    open_brackets = cleaned.count('[') - cleaned.count(']')

    # Close arrays first, then objects
    cleaned += ']' * max(0, open_brackets)
    cleaned += '}' * max(0, open_braces)

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # More aggressive: find the outermost { and try to parse just that
    brace_start = raw.find('{')
    if brace_start >= 0:
        attempt = raw[brace_start:]
        attempt = attempt.rstrip().rstrip(',')
        if attempt.count('"') % 2 != 0:
            last_q = attempt.rfind('"')
            attempt = attempt[:last_q]
            last_good = max(attempt.rfind(','), attempt.rfind('['), attempt.rfind('{'))
            if last_good > 0:
                attempt = attempt[:last_good + 1]
        attempt = attempt.rstrip().rstrip(',')
        ob = attempt.count('{') - attempt.count('}')
        obrk = attempt.count('[') - attempt.count(']')
        attempt += ']' * max(0, obrk)
        attempt += '}' * max(0, ob)
        try:
            return json.loads(attempt)
        except json.JSONDecodeError:
            pass

    return None


def _parse_pdf_with_vision(
    pdf_path: Path,
    openai_api_key: str | None = None,
    model: str = "gpt-4o",
) -> dict:
    """
    Parse a scanned PDF roster using GPT-5 vision.

    Renders each page, sends to GPT-5, merges results.
    """
    from openai import OpenAI

    # Suppress noisy HTTP debug logs from the OpenAI / httpx client
    for _noisy in ("openai", "httpx", "httpcore"):
        logging.getLogger(_noisy).setLevel(logging.WARNING)

    api_key = openai_api_key or os.environ.get("OPENAI_API_KEY")
    if not api_key:
        raise ValueError(
            "OpenAI API key required for PDF roster parsing. "
            "Pass --openai-key or set OPENAI_API_KEY environment variable."
        )

    client = OpenAI(api_key=api_key)
    pages = _render_pdf_pages(pdf_path, dpi=300)
    logger.info("Rendered %d PDF pages from %s", len(pages), pdf_path.name)

    all_classes: dict[str, dict] = {}

    for page_num, png_bytes in enumerate(pages, start=1):
        logger.info("  Sending page %d/%d to %s ...", page_num, len(pages), model)

        content = [
            {"type": "text", "text": _PDF_ROSTER_PROMPT + f"\n\nThis is page {page_num} of {len(pages)}."},
            {"type": "image_url", "image_url": {
                "url": _image_to_data_url(png_bytes),
                "detail": "high",
            }},
        ]

        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You extract structured data from school documents. Return only valid JSON."},
                {"role": "user", "content": content},
            ],
            #temperature=0.0,
            max_tokens=8192,
            response_format={"type": "json_object"}

        )

        raw = resp.choices[0].message.content or ""
        finish_reason = resp.choices[0].finish_reason
        # Strip markdown fences if present
        raw = re.sub(r"^```(?:json)?\s*", "", raw.strip())
        raw = re.sub(r"\s*```$", "", raw)

        if finish_reason == "length":
            logger.warning("  Page %d: response was truncated (max_tokens). "
                           "Attempting to repair JSON...", page_num)

        try:
            page_data = json.loads(raw)
        except json.JSONDecodeError:
            # Try to repair truncated JSON by closing open structures
            repaired = _repair_truncated_json(raw)
            if repaired is not None:
                page_data = repaired
                logger.info("  Page %d: successfully repaired truncated JSON", page_num)
            else:
                logger.warning("  Page %d: failed to parse GPT response as JSON", page_num)
                logger.debug("  Raw response (first 500 chars): %s", raw[:500])
                continue

        page_classes = page_data.get("classes", {})
        for cls_letter, cls_data in page_classes.items():
            cls_letter = cls_letter.upper()
            if cls_letter not in all_classes:
                all_classes[cls_letter] = {
                    "teacher": cls_data.get("teacher"),
                    "students": [],
                }
            # Merge students (avoid duplicates by attendance number)
            existing_nums = {s["number"] for s in all_classes[cls_letter]["students"]}
            for student in cls_data.get("students", []):
                num = student.get("number")
                if num is not None and num not in existing_nums:
                    all_classes[cls_letter]["students"].append({
                        "number": int(num),
                        "name": student.get("name", ""),
                        "furigana": student.get("furigana", ""),
                        "gender": student.get("gender", ""),
                    })
                    existing_nums.add(num)

            # Update teacher if not set
            if not all_classes[cls_letter]["teacher"] and cls_data.get("teacher"):
                all_classes[cls_letter]["teacher"] = cls_data["teacher"]

        logger.info("  Page %d: extracted %d classes", page_num,
                    len(page_classes))

    # Sort students by number within each class
    for cls_data in all_classes.values():
        cls_data["students"].sort(key=lambda s: s["number"])
        cls_data["student_count"] = len(cls_data["students"])

    return {
        "year": 3,  # Will be overridden or inferred
        "classes": all_classes,
    }


# ──────────────────────────────────────────────────────────────────
# Public API
# ──────────────────────────────────────────────────────────────────

def parse_roster(
    roster_path: str | Path,
    school_name: str | None = None,
    openai_api_key: str | None = None,
    vision_model: str = "gpt-4o",
) -> dict:
    """
    Parse a school roster file (.xlsx or .pdf) into a normalised dictionary.

    Parameters
    ----------
    roster_path : path to .xlsx or .pdf file
    school_name : optional override; auto-detected from filename if omitted
    openai_api_key : required for PDF parsing (GPT-4o vision)
    vision_model : OpenAI model for PDF vision extraction (default: gpt-4o)

    Returns
    -------
    dict  – see module docstring for schema
    """
    roster_path = Path(roster_path)
    if not roster_path.exists():
        raise FileNotFoundError(f"Roster file not found: {roster_path}")

    if school_name is None:
        # Strip prefix numbers and extension
        stem = roster_path.stem
        stem = re.sub(r"^\d+", "", stem).strip("_ ")
        school_name = stem or "Unknown"

    ext = roster_path.suffix.lower()

    if ext == ".pdf":
        # ── PDF: GPT-5 vision extraction ──
        logger.info("Detected PDF roster — using %s vision for extraction", vision_model)
        result = _parse_pdf_with_vision(
            roster_path,
            openai_api_key=openai_api_key,
            model=vision_model,
        )
        result["format_detected"] = f"PDF (via {vision_model} vision)"

    elif ext in (".xlsx", ".xls", ".xlsm"):
        # ── Excel parsing ──
        wb = openpyxl.load_workbook(str(roster_path), data_only=True, read_only=False)
        try:
            if _detect_format_b(wb):
                result = _parse_format_b(wb)
                result["format_detected"] = "B (paired-class sheets)"
            elif _detect_format_a(wb):
                result = _parse_format_a(wb)
                result["format_detected"] = "A (wide HR-block)"
            else:
                result = _parse_generic(wb)
                result["format_detected"] = "generic (auto-detect)"
        finally:
            wb.close()

    else:
        raise ValueError(f"Unsupported roster file format: {ext}. Use .xlsx or .pdf")

    result["school"] = school_name
    result["source_file"] = str(roster_path)

    # ── Validation ──
    warnings = []
    for cls_letter, cls_data in result.get("classes", {}).items():
        students = cls_data.get("students", [])
        numbers = [s["number"] for s in students]
        if numbers:
            expected = list(range(1, max(numbers) + 1))
            missing = sorted(set(expected) - set(numbers))
            if missing:
                warnings.append(f"Class {cls_letter}: missing attendance numbers {missing}")
            dupes = sorted(n for n in numbers if numbers.count(n) > 1)
            if dupes:
                warnings.append(f"Class {cls_letter}: duplicate numbers {list(set(dupes))}")
    result["warnings"] = warnings

    return result


def save_roster_json(roster: dict, output_path: str | Path) -> Path:
    """Write roster dict to a JSON file."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(roster, f, ensure_ascii=False, indent=2)
    return output_path


def get_valid_numbers(roster: dict, class_letter: str) -> set[int]:
    """Return the set of valid attendance numbers for a class."""
    cls = roster.get("classes", {}).get(class_letter, {})
    return {s["number"] for s in cls.get("students", [])}


def lookup_student(roster: dict, class_letter: str, number: int) -> dict | None:
    """Look up a student by class and attendance number."""
    cls = roster.get("classes", {}).get(class_letter, {})
    for s in cls.get("students", []):
        if s["number"] == number:
            return s
    return None


In [ ]:
src = "Photography\\Client Sample Image\\20260224　個人写真_レイヤー概念など\\26篠崎高ネーム.pdf"
api_key = ""
dst = src.rsplit(".", 1)[0] + "_roster.json"

roster = parse_roster(src, openai_api_key=api_key)
save_roster_json(roster, dst)

total = sum(len(c["students"]) for c in roster["classes"].values())
print(f"✅ Parsed: {roster['school']}")
print(f"   Format : {roster['format_detected']}")
print(f"   Classes: {len(roster['classes'])}  ({', '.join(sorted(roster['classes']))})")
print(f"   Students: {total}")
if roster["warnings"]:
    for w in roster["warnings"]:
        print(f"   ⚠️  {w}")
print(f"   Saved  : {dst}")

INFO │ Detected PDF roster — using gpt-4o vision for extraction
INFO │ Rendered 2 PDF pages from 26篠崎高ネーム.pdf
INFO │   Sending page 1/2 to gpt-4o ...
INFO │   Page 1: extracted 4 classes
INFO │   Sending page 2/2 to gpt-4o ...
INFO │   Page 2: extracted 3 classes


✅ Parsed: 篠崎高ネーム
   Format : PDF (via gpt-4o vision)
   Classes: 7  (1, 2, 3, 4, 5, 6, 7)
   Students: 280
   Saved  : D:\career\LTID\Photography\Client Sample Image\20260224　個人写真_レイヤー概念など\26篠崎高ネーム_roster.json


In [ ]:
# Display extracted data
# Assuming 'roster' contains the parsed data from the previous cell
import json

# Pretty-print the extracted data
print(json.dumps(roster, ensure_ascii=False, indent=2))

{
  "year": 3,
  "classes": {
    "1": {
      "teacher": "古宮 百合里",
      "students": [
        {
          "number": 1,
          "name": "秋山  真実",
          "furigana": "あきやま  まな",
          "gender": ""
        },
        {
          "number": 2,
          "name": "芦部 裕斗",
          "furigana": "あしべ ゆうと",
          "gender": ""
        },
        {
          "number": 3,
          "name": "荒 勇樹",
          "furigana": "あら  ゆうき",
          "gender": ""
        },
        {
          "number": 4,
          "name": "飯塚 一輝",
          "furigana": "いいづか かずき",
          "gender": ""
        },
        {
          "number": 5,
          "name": "池田 陽利",
          "furigana": "いけだ  はると",
          "gender": ""
        },
        {
          "number": 6,
          "name": "池田 樹利",
          "furigana": "いけだ じゅり",
          "gender": ""
        },
        {
          "number": 7,
          "name": "磯部 慶太郎",
          "furigana": "いそべ けいたろう",
          "gender": ""
        },
        {
       

## Next Steps

1. Run the example code with a sample PDF file.
2. Review the output JSON file and ensure it matches the expected schema.
3. Suggest any improvements or alternative approaches for parsing the rosters.

Feel free to reach out if you have any questions or need further clarification.